# MORQEX geometry workflow / MORQEX几何建模

256³ original voxels, live2D/3D output and full streamedTet4. Execute a copy outside source.
原始256³体素、实时二维/三维与完整分块四面体；请在源码外执行副本。

In [ ]:
from pathlib import Path
import sys,os
SOURCE=Path('/home/zephyrain/lw/MORQEX')
sys.path.insert(0,str(SOURCE/'src'))
WORK=Path('/home/zephyrain/lw/morph_workspace/microelectrode_gallery/notebook_example')
WORK.mkdir(parents=True,exist_ok=True)
import numpy as np
from morqex.modeling import from_regions,Shape,from_shapes


## Live view / 实时预览
The same display handle is updated during construction.
生成过程复用同一显示句柄。

In [ ]:
blank=from_regions(np.zeros((256,)*3,np.uint8),spacing=(0.5e-6,)*3)
view=blank.show(maximum_3d_size=40,height=680,title='Live geometry / 实时几何')


In [ ]:
rng=np.random.default_rng(20261127)
objects=[]
for k in range(18):
    angle=float(rng.uniform(-0.8,0.8))
    R=np.array([[np.cos(angle),0,np.sin(angle)],[0,1,0],[-np.sin(angle),0,np.cos(angle)]])
    objects.append(Shape('capsule',1,tuple(rng.uniform(15e-6,113e-6,3)),(4e-6,4e-6,float(rng.uniform(12e-6,30e-6))),tuple(map(tuple,R)),0.2))
model=from_shapes(objects,shape=(256,)*3,spacing=(0.5e-6,)*3,progress=view.update,progress_every=6)
view.update(model)
view.save_html(WORK/'live_geometry.html')


## Full tetrahedral mesh / 完整体四面体网格
The full256³ array is meshed; displayLOD does not enter the mesh calculation.
对完整256³数组生成网格，预览LOD不参与计算。

In [ ]:
target=WORK/'mesh.h5'
if not target.exists():
    artifact=model.mesh_to(target,regions=[1],slab_depth=2)
    print(artifact)
else:
    print('Existing mesh retained / 保留已有网格:',target)


In [ ]:
model.save(WORK/'voxels.h5')
print('HTML:',WORK/'live_geometry.html')
print('ParaView:',WORK/'mesh.xdmf')


In [ ]:
from morqex.modeling import MeshArtifact
import h5py
with h5py.File(WORK/'mesh.h5') as f:
    saved=MeshArtifact(WORK/'mesh.h5',WORK/'mesh.xdmf',int(f.attrs['node_count']),int(f.attrs['tetrahedra']),(1,))
mesh_view=saved.show(first_cell=0,cell_count=600,height=620)
mesh_view.save_html(WORK/'actual_mesh.html')
